# Simulação do Protocolo de Árvore Geradora (STP / RSTP - IEEE 802.1w)
### Topologia em Anel Redundante da Rede Industrial SCADA da Esteira Seletora (FMS)
**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta  
**Ambiente:** Rede Ethernet Industrial (PROFINET / Modbus TCP) interligando a Sala de Controle SCADA, o Painel Central do CLP Siemens S7-1500 (`CLP-00`) e as remotas de campo dos Setores 100, 200 e 300  

---

## 🎯 Objetivos do Notebook
1. **Modelagem Topológica em Grafo Ponderado:** Construir o grafo $G = (V, E, w)$ representando os switches industriais gerenciáveis e os enlaces de fibra óptica e cabo metálico sob os custos da norma **IEEE 802.1w**.
2. **Eleição Determinística do *Root Bridge*:** Implementar a comparação lexicográfica de *Bridge ID* $(\text{Priority}, \text{MAC})$.
3. **Cálculo da Árvore Geradora e Papéis de Porta:** Determinar automaticamente as portas *Root Port (RP)*, *Designated Port (DP)* e *Alternate Port (AP)*, bloqueando logicamente os ciclos para evitar *broadcast storms*.
4. **Simulação de Falha Física (*Link Down*) e Reconvergência:** Simular o rompimento da fibra óptica principal de $10\text{ Gbps}$ e auditar a reconvergência instantânea com ativação da rota de backup.
5. **Auditoria Rigorosa com Asserções (`assert`):** Comprovar matematicamente que a topologia ativa satisfaz $|E_T| = |V| - 1 = 4$, garantindo que $T$ é conexo e acíclico.
6. **Visualização Gráfica Comparativa:** Plotar a rede em operação normal e após a falha de link com destaque de estados de porta.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Tuple, Any

# Configurações estéticas de plotagem
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110

print("✅ Módulos carregados com sucesso para a simulação RSTP da FMS.")

---
## 1. Definição da Rede Industrial da Manufatura Flexível

Configuramos os switches gerenciáveis de cada setor com suas prioridades IEEE 802.1w para garantir que o switch da Sala de Controle (`SW-SCADA`) vença a eleição como Root Bridge primário e o switch do Painel do CLP (`SW-CLP00`) atue como Root Bridge de backup.

In [ ]:
# 1. VÉRTICES (Switches Industriais da Esteira Seletora)
# Bridge ID (BID) = (Bridge_Priority, MAC_Address)
switches_fms = {
    "SW-SCADA": {"nome": "Sala de Controle SCADA-Core", "priority": 4096,  "mac": "00:0E:CF:00:00:01"},
    "SW-CLP00": {"nome": "Painel Central CLP Siemens",   "priority": 8192,  "mac": "00:0E:CF:00:00:02"},
    "SW-S100":  {"nome": "Setor 100: Silo e Esteira",    "priority": 32768, "mac": "00:0E:CF:00:00:03"},
    "SW-S200":  {"nome": "Setor 200: Desvios e Túnel",  "priority": 32768, "mac": "00:0E:CF:00:00:04"},
    "SW-S300":  {"nome": "Setor 300: Loteamento",       "priority": 32768, "mac": "00:0E:CF:00:00:05"},
}

# 2. ARESTAS (Enlaces Físicos e Custos Padrão IEEE 802.1w)
# 10 Gbps = 2.000 | 1 Gbps = 20.000 | 100 Mbps = 200.000
enlaces_fms = [
    ("SW-SCADA", "SW-CLP00", {"cost": 2000,   "tipo": "10 Gbps (Fibra Backbone)"}),
    ("SW-CLP00", "SW-S100",  {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel)"}),
    ("SW-S100",  "SW-S200",  {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel)"}),
    ("SW-S200",  "SW-S300",  {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel)"}),
    ("SW-S300",  "SW-SCADA", {"cost": 20000,  "tipo": "1 Gbps (Fibra Anel Fechamento)"}),
    # Enlace secundário redundante em cabo de cobre Cat6A
    ("SW-CLP00", "SW-S200",  {"cost": 200000, "tipo": "100 Mbps (Cabo Cobre Backup)"}),
]

# Construção do Grafo Físico G = (V, E, w)
G_fms = nx.Graph()
for sw_id, attrs in switches_fms.items():
    G_fms.add_node(sw_id, **attrs)
G_fms.add_edges_from(enlaces_fms)

num_nos = G_fms.number_of_nodes()
num_arestas = G_fms.number_of_edges()
num_ciclos = num_arestas - num_nos + 1

print(f"Grafo da Rede Industrial FMS criado com |V| = {num_nos} nós e |E| = {num_arestas} enlaces.")
print(f"Número ciclomático (laços fechados a serem eliminados): {num_ciclos} ciclos independentes.")

---
## 2. Implementação do Motor de Convergência do RSTP (IEEE 802.1w)

O algoritmo executa:
1. **Eleição do Root Bridge:** Seleciona o nó com menor tupla `(priority, mac)`.
2. **Cálculo dos Custos de Caminho da Raiz (*Root Path Cost*):** Determina as menores distâncias ponderadas a partir do Root Bridge.
3. **Determinação dos Papéis de Porta:**
   - **Root Port (RP):** Porta que oferece o caminho mais curto até o Root Bridge (estado *Forwarding*).
   - **Designated Port (DP):** Porta com melhor métrica no segmento local (estado *Forwarding*).
   - **Alternate Port (AP):** Porta descartada por redundância para prevenção de loops (estado *Discarding*).

In [ ]:
def executar_rstp(grafo: nx.Graph, switches: Dict[str, dict]) -> Tuple[str, Dict[str, int], List[dict], nx.Graph]:
    """
    Simula a convergência completa do protocolo RSTP (IEEE 802.1w).
    Retorna o Root Bridge, os custos de caminho, a tabela de papéis de porta e a Spanning Tree ativa.
    """
    # 1. Eleição do Root Bridge (Menor tupla: prioridade, mac)
    root_bridge = min(switches.keys(), key=lambda sw: (switches[sw]['priority'], switches[sw]['mac']))
    
    # 2. Cálculo dos custos de caminho mínimos a partir do Root Bridge
    path_costs = nx.single_source_dijkstra_path_length(grafo, root_bridge, weight='cost')
    shortest_paths = nx.single_source_dijkstra_path(grafo, root_bridge, weight='cost')
    
    # Construção da Árvore Geradora T = (V, E_T)
    T = nx.Graph()
    for node in grafo.nodes():
        T.add_node(node, **switches[node])
        
    # Identificar as Root Ports de cada nó não-raiz
    root_ports: Dict[str, str] = {}  # nó -> vizinho através da RP
    for node, path in shortest_paths.items():
        if node != root_bridge:
            vizinho_upstream = path[-2]
            root_ports[node] = vizinho_upstream
            custo_aresta = grafo[node][vizinho_upstream]['cost']
            T.add_edge(node, vizinho_upstream, cost=custo_aresta)
            
    # 3. Classificação das portas em cada enlace físico
    tabela_portas = []
    for u, v, data in grafo.edges(data=True):
        custo = data['cost']
        enlace_str = f"{u} <--> {v}"
        
        # Verificar se a aresta pertence à Árvore Geradora ativa
        if T.has_edge(u, v):
            status_enlace = "ATIVO (Encaminhamento / Forwarding)"
            # Identificar quem é RP e quem é DP
            if root_ports.get(u) == v:
                papel_u = "Root Port (RP) [Forwarding]"
                papel_v = "Designated Port (DP) [Forwarding]"
            elif root_ports.get(v) == u:
                papel_u = "Designated Port (DP) [Forwarding]"
                papel_v = "Root Port (RP) [Forwarding]"
            else:
                papel_u = "Designated Port (DP) [Forwarding]"
                papel_v = "Designated Port (DP) [Forwarding]"
        else:
            status_enlace = "BLOQUEADO (Loop Prevention / Discarding)"
            # Em arestas bloqueadas, o nó com melhor métrica é DP e o outro é Alternate Port (AP)
            cost_u = (path_costs[u], switches[u]['priority'], switches[u]['mac'])
            cost_v = (path_costs[v], switches[v]['priority'], switches[v]['mac'])
            if cost_u < cost_v:
                papel_u = "Designated Port (DP) [Forwarding]"
                papel_v = "Alternate Port (AP) [Discarding (Bloqueada)]"
            else:
                papel_u = "Alternate Port (AP) [Discarding (Bloqueada)]"
                papel_v = "Designated Port (DP) [Forwarding]"
                
        tabela_portas.append({
            "Enlace Físico": enlace_str,
            "Custo IEEE": custo,
            "Tipo Enlace": data.get('tipo', 'Fibra'),
            f"Papel em {u}": papel_u,
            f"Papel em {v}": papel_v,
            "Status Operacional": status_enlace
        })
        
    return root_bridge, path_costs, tabela_portas, T

print("✅ Motor de simulação RSTP compilado com sucesso.")

---
## 3. Avaliação da Operação Normal da Rede e Auditoria com `assert`

Executamos o RSTP na topologia íntegra da Célula de Manufatura Flexível.

In [ ]:
root_normal, custos_normal, portas_normal, T_normal = executar_rstp(G_fms, switches_fms)

df_portas_normal = pd.DataFrame(portas_normal)

print("=== CONVERGÊNCIA RSTP: OPERAÇÃO NORMAL ===")
print(f"🏆 Root Bridge Eleito : {root_normal} (Prioridade: {switches_fms[root_normal]['priority']})")
print("\nCustos de Caminho até o Root Bridge:")
for sw, c in custos_normal.items():
    print(f"  - {sw:10s}: {c:6d} [{switches_fms[sw]['nome']}]")

print("\nTabela de Estados e Papéis de Porta:")
display(df_portas_normal[["Enlace Físico", "Custo IEEE", "Status Operacional"]])

# ==========================================================================
# ASSERÇÕES DE VALIDAÇÃO FORMAL (OPERAÇÃO NORMAL)
# ==========================================================================
# 1. Verificar eleição do Root Bridge correto
assert root_normal == "SW-SCADA", f"Falha na eleição: {root_normal} eleito em vez de SW-SCADA!"

# 2. Verificar que o subgrafo ativo é uma Árvore Geradora estrita
assert nx.is_tree(T_normal) == True, "Falha: Subgrafo gerador ativo não é uma árvore acíclica!"
assert T_normal.number_of_nodes() == 5, "A árvore deve abranger todos os 5 switches da planta!"
assert T_normal.number_of_edges() == 4, "Uma árvore com 5 vértices deve ter exatamente 4 arestas ativas!"

# 3. Verificar que exatamente 2 enlaces foram colocados em Discarding (6 - 4 = 2)
bloqueados = [p for p in portas_normal if "BLOQUEADO" in p["Status Operacional"]]
assert len(bloqueados) == 2, f"Número incorreto de portas bloqueadas: {len(bloqueados)} (esperado: 2)"

# 4. Verificar que o enlace de 10 Gbps (Backbone SCADA-CLP) está ATIVO
enlace_backbone = next(p for p in portas_normal if "SW-SCADA <--> SW-CLP00" in p["Enlace Físico"])
assert "ATIVO" in enlace_backbone["Status Operacional"], "O enlace de 10G principal não pode ser bloqueado!"

print("\n✅ Todas as asserções de operação normal foram aprovadas com 100% de conformidade!")

---
## 4. Simulação de Falha Física de Enlace (*Link Failure*) e Reconvergência Rápida

Simulamos o corte acidental da fibra óptica principal de $10\text{ Gbps}$ entre a Sala de Controle (`SW-SCADA`) e o Painel do CLP (`SW-CLP00`). O RSTP deve detectar a falha em milissegundos e desbloquear automaticamente uma das portas *Alternate* para restabelecer a conectividade de toda a planta.

In [ ]:
# Criar cópia do grafo físico e simular corte do enlace (SW-SCADA, SW-CLP00)
G_falha = G_fms.copy()
G_falha.remove_edge("SW-SCADA", "SW-CLP00")

# Executar nova convergência RSTP sobre o grafo sob falha
root_falha, custos_falha, portas_falha, T_falha = executar_rstp(G_falha, switches_fms)

df_portas_falha = pd.DataFrame(portas_falha)

print("=== CONVERGÊNCIA RSTP: PÓS-FALHA DE LINK (ROMPIMENTO DE FIBRA) ===")
print(f"🏆 Root Bridge Mantido : {root_falha}")
print("\nNovos Custos de Caminho até o Root Bridge:")
for sw, c in custos_falha.items():
    print(f"  - {sw:10s}: {c:6d} [{switches_fms[sw]['nome']}]")

print("\nTabela de Portas Reconfigurada Automaticamente:")
display(df_portas_falha[["Enlace Físico", "Custo IEEE", "Status Operacional"]])

# ==========================================================================
# ASSERÇÕES DE VALIDAÇÃO SOB FALHA DE REDE
# ==========================================================================
# 1. Verificar conectividade total pós-falha
assert nx.is_connected(T_falha) == True, "Falha Crítica: Rede fragmentada após rompimento da fibra!"
assert nx.is_tree(T_falha) == True, "O subgrafo gerador deve permanecer como árvore acíclica!"
assert T_falha.number_of_edges() == 4, "A nova árvore deve possuir exatamente 4 arestas ativas!"

# 2. Verificar que o CLP-00 agora escoa tráfego através do Setor 100/200/300
assert custos_falha["SW-CLP00"] > custos_normal["SW-CLP00"], "Custo para o CLP deve aumentar devido ao desvio pelo anel"

# 3. Verificar que o enlace SW-S200 <--> SW-S300 foi DESBLOQUEADO (assumiu tráfego)
enlace_revertido = next(p for p in portas_falha if "SW-S200 <--> SW-S300" in p["Enlace Físico"])
assert "ATIVO" in enlace_revertido["Status Operacional"], "A porta Alternate deveria ter comutado para Forwarding!"

print("\n✅ Reconvergência RSTP validada com 100% de sucesso sob injeção de falha física!")

---
## 5. Visualização Comparativa da Topologia de Rede Industrial

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Layout visual dos switches simulando a posição física na fábrica
pos_switches = {
    "SW-SCADA": (1.5, 3.5),   # Sala de Controle (Norte)
    "SW-CLP00": (0.5, 2.0),   # Painel Principal (Oeste)
    "SW-S100":  (1.0, 0.5),   # Infeed (Sudoeste)
    "SW-S200":  (2.5, 0.5),   # Triagem (Sudeste)
    "SW-S300":  (3.0, 2.0),   # Loteamento (Leste)
}

# ==========================================================================
# SUBPLOT 1: OPERAÇÃO NORMAL
# ==========================================================================
ax1 = axes[0]
ax1.set_title("A. Operação Normal da Rede SCADA (RSTP Ativo)", fontsize=12, fontweight='bold')

# Desenhar enlaces ativos da árvore em verde sólido
arestas_ativas = list(T_normal.edges())
nx.draw_networkx_edges(G_fms, pos_switches, edgelist=arestas_ativas, width=3.5, edge_color='#10b981', ax=ax1, label='Enlaces Ativos (Forwarding)')

# Desenhar enlaces bloqueados em vermelho tracejado
arestas_bloqueadas = [e for e in G_fms.edges() if not T_normal.has_edge(*e)]
nx.draw_networkx_edges(G_fms, pos_switches, edgelist=arestas_bloqueadas, width=2.5, edge_color='#ef4444', style='--', ax=ax1, label='Enlaces Bloqueados (Discarding)')

# Nós (Switches)
cores_nos_normal = ['#eab308' if n == root_normal else '#1e293b' for n in G_fms.nodes()]
nx.draw_networkx_nodes(G_fms, pos_switches, node_size=1600, node_color=cores_nos_normal, edgecolors='black', linewidths=1.5, ax=ax1)
nx.draw_networkx_labels(G_fms, pos_switches, font_size=9, font_color='white', font_weight='bold', ax=ax1)

ax1.legend(loc='lower center', framealpha=0.9, fontsize=9)
ax1.axis('off')

# ==========================================================================
# SUBPLOT 2: PÓS-FALHA DE LINK E RECONVERGÊNCIA
# ==========================================================================
ax2 = axes[1]
ax2.set_title("B. Reconvergência Rápida após Rompimento de Fibra (10G Link Down)", fontsize=12, fontweight='bold')

# Desenhar enlace rompido em cinza com 'X' vermelho
nx.draw_networkx_edges(G_fms, pos_switches, edgelist=[("SW-SCADA", "SW-CLP00")], width=2.0, edge_color='#94a3b8', style=':', ax=ax2, label='Enlace Rompido (Link Down)')

# Desenhar novos enlaces ativos
arestas_ativas_falha = list(T_falha.edges())
nx.draw_networkx_edges(G_falha, pos_switches, edgelist=arestas_ativas_falha, width=3.5, edge_color='#10b981', ax=ax2, label='Nova Árvore Ativa (Reconvergida)')

# Arestas ainda bloqueadas
arestas_bloq_falha = [e for e in G_falha.edges() if not T_falha.has_edge(*e)]
nx.draw_networkx_edges(G_falha, pos_switches, edgelist=arestas_bloq_falha, width=2.5, edge_color='#ef4444', style='--', ax=ax2, label='Backup Secundário Bloqueado')

# Nós
cores_nos_falha = ['#eab308' if n == root_falha else '#1e293b' for n in G_falha.nodes()]
nx.draw_networkx_nodes(G_falha, pos_switches, node_size=1600, node_color=cores_nos_falha, edgecolors='black', linewidths=1.5, ax=ax2)
nx.draw_networkx_labels(G_falha, pos_switches, font_size=9, font_color='white', font_weight='bold', ax=ax2)

ax2.legend(loc='lower center', framealpha=0.9, fontsize=9)
ax2.axis('off')

plt.tight_layout()
plt.show()

---
## 6. Conclusões de Engenharia de Redes Industriais

1. **Prevenção Confiável de Loops:** Em operação normal, o algoritmo RSTP bloqueou deterministicamente os 2 ciclos físicos do anel, eliminando o risco de tempestades de broadcast (*broadcast storms*) e garantindo determinismo na comunicação PROFINET.
2. **Resiliência e Tolerância a Falhas:** Sob corte físico do enlace central de $10\text{ Gbps}$, a porta *Alternate* do anel comutou para *Forwarding* instantaneamente, preservando a supervisão e o controle contínuo da esteira seletora sem interrupção de produção.